# 03 Parent Child Chunking

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `03-parent-child-chunking.ipynb`

In [ ]:
# Start coding here

In [39]:
# =====================================================
# Notebook 03
# Parent Child Chunking
# =====================================================

import json
import uuid

import pandas as pd

In [40]:
with open("artifacts/document_tree.json", "r", encoding="utf-8") as file:

    document_tree = json.load(file)

print(f"Loaded {len(document_tree)} documents")

Loaded 3 documents


In [41]:
first_doc = list(document_tree.keys())[0]

document_tree[first_doc]

[{'type': 'title', 'content': 'AI Platform Roadmap'},
 {'type': 'heading', 'content': 'Search Infrastructure'},
 {'type': 'heading', 'content': 'Retrieval Systems'},
 {'type': 'heading', 'content': 'Deployment Roadmap'},
 {'type': 'paragraph', 'content': 'Vector search infrastructure rollout.'},
 {'type': 'paragraph', 'content': 'Search latency improved significantly.'},
 {'type': 'paragraph', 'content': 'New indexing pipeline deployed.'},
 {'type': 'paragraph', 'content': 'Hybrid retrieval implementation.'},
 {'type': 'paragraph', 'content': 'Semantic search quality improved.'},
 {'type': 'paragraph', 'content': 'Knowledge base coverage expanded.'},
 {'type': 'paragraph', 'content': 'Cross encoder deployment planned.'},
 {'type': 'paragraph', 'content': 'Enterprise RAG platform launch Q3.'},
 {'type': 'paragraph', 'content': 'Agentic workflows under evaluation.'},
 {'type': 'table', 'content': 'Component | Status'},
 {'type': 'table', 'content': 'Vector Search | Production'},
 {'type'

In [42]:
def build_document_text(elements):

    text = ""

    for element in elements:

        text += element["content"] + "\n"

    return text

In [43]:
sample_text = build_document_text(document_tree[first_doc])

print(sample_text)

AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrastructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retrieval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encoder deployment planned.
Enterprise RAG platform launch Q3.
Agentic workflows under evaluation.
Component | Status
Vector Search | Production
Hybrid Retrieval | Testing
Cross Encoder | Planned



In [44]:
PARENT_SIZE = 300
CHILD_SIZE = 100

In [45]:
def create_parent_chunks(text, parent_size=300):

    chunks = []

    start = 0

    while start < len(text):

        end = start + parent_size

        chunk = text[start:end]

        chunks.append(chunk)

        start = end

    return chunks

In [46]:
parents = create_parent_chunks(sample_text)

len(parents)

2

In [47]:
print(parents[0])

AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrastructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retrieval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encod


In [48]:
def create_child_chunks(parent_text, child_size=100):

    chunks = []

    start = 0

    while start < len(parent_text):

        end = start + child_size

        chunk = parent_text[start:end]

        chunks.append(chunk)

        start = end

    return chunks

In [49]:
children = create_child_chunks(parents[0])

len(children)

3

In [50]:
for idx, child in enumerate(children):

    print(f"\nChild {idx+1}")

    print("-" * 40)

    print(child)


Child 1
----------------------------------------
AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrast

Child 2
----------------------------------------
ructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retri

Child 3
----------------------------------------
eval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encod


In [51]:
parent_schema = {"parent_id": "", "document_id": "", "content": ""}

parent_schema

{'parent_id': '', 'document_id': '', 'content': ''}

In [52]:
child_schema = {"child_id": "", "parent_id": "", "document_id": "", "content": ""}

child_schema

{'child_id': '', 'parent_id': '', 'document_id': '', 'content': ''}

In [53]:
parent_records = []

for doc_id, elements in document_tree.items():

    text = build_document_text(elements)

    parent_chunks = create_parent_chunks(text)

    for chunk in parent_chunks:

        parent_records.append(
            {"parent_id": str(uuid.uuid4()), "document_id": doc_id, "content": chunk}
        )

In [54]:
parents_df = pd.DataFrame(parent_records)

parents_df.head()

,parent_id,document_id,content
0,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...
1,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,er deployment planned.\nEnterprise RAG platfor...
2,9cb41e2d-b979-4c89-a091-01db42c3748e,b638264b-b788-442e-a09d-19176466fba8,Annual Financial Report 2026\nRevenue Performa...
3,7cdee370-fba0-4bc7-9fe1-a39230162073,b638264b-b788-442e-a09d-19176466fba8,oud division grew by 35%.\nAI products generat...
4,fd7bb5e1-9ec6-470a-8726-bbdc04fcc6dc,31516a79-2cf5-4080-ac66-66b338d6ebff,Employee Benefits 2026\nHealth Insurance\nDent...


In [55]:
child_records = []

for _, parent in parents_df.iterrows():

    child_chunks = create_child_chunks(parent["content"])

    for chunk in child_chunks:

        child_records.append(
            {
                "child_id": str(uuid.uuid4()),
                "parent_id": parent["parent_id"],
                "document_id": parent["document_id"],
                "content": chunk,
            }
        )

In [56]:
children_df = pd.DataFrame(child_records)

children_df.head()

,child_id,parent_id,document_id,content
0,5f2b6fb4-0d03-4817-900a-945162a178c2,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...
1,047bf5df-d450-4bbe-92c2-2396ed3a7b30,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,ructure rollout.\nSearch latency improved sign...
2,b400d323-ca7c-4813-ac4e-a2ba69786f3f,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,eval implementation.\nSemantic search quality ...
3,e68fd4e0-8796-416e-b24b-46ccae0f8d1d,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,er deployment planned.\nEnterprise RAG platfor...
4,e001567a-7134-4290-9ebb-e7e07e069dcb,a1c5bb38-4de7-4ca4-805a-7e3fc0419867,1fe99400-06e1-4964-956b-3861687e7da4,ent | Status\nVector Search | Production\nHybr...


In [57]:
sample_parent = parents_df.iloc[0]

sample_parent_id = sample_parent["parent_id"]

children_df[children_df["parent_id"] == sample_parent_id]

,child_id,parent_id,document_id,content
0,5f2b6fb4-0d03-4817-900a-945162a178c2,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...
1,047bf5df-d450-4bbe-92c2-2396ed3a7b30,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,ructure rollout.\nSearch latency improved sign...
2,b400d323-ca7c-4813-ac4e-a2ba69786f3f,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,eval implementation.\nSemantic search quality ...


In [58]:
sample_child = children_df.iloc[0]

parent_id = sample_child["parent_id"]

parent_context = parents_df[parents_df["parent_id"] == parent_id]

parent_context

,parent_id,document_id,content
0,7d09d047-2858-43c4-bf8b-d40519129087,1fe99400-06e1-4964-956b-3861687e7da4,AI Platform Roadmap\nSearch Infrastructure\nRe...


In [59]:
print("Parents:", len(parents_df))

print("Children:", len(children_df))

Parents: 6
Children: 16


In [60]:
parents_df.to_csv("artifacts/parent_chunks.csv", index=False)

In [61]:
children_df.to_csv("artifacts/child_chunks.csv", index=False)

In [62]:
# !pip install langchain
# !pip install langchain-community
# !pip install langchain-text-splitters

In [63]:
import langchain

print(langchain)
print(langchain.__file__)
print(langchain.__version__)

<module 'langchain' from 'd:\\Books\\projects-nlp-transformers-learning\\.projectnlps\\Lib\\site-packages\\langchain\\__init__.py'>
d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\langchain\__init__.py
1.3.10


In [64]:
# !pip install langchain-community
# !pip install langchain-core

In [65]:
!pip show langchain

Name: langchain
Version: 1.3.10
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: D:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [67]:
def retrieve_parent_document(
    child_id,
    children_df,
    parents_df,
):
    child_record = children_df[children_df["child_id"] == child_id].iloc[0]

    parent_id = child_record["parent_id"]

    parent_record = parents_df[parents_df["parent_id"] == parent_id].iloc[0]

    return parent_record

In [68]:
sample_child_id = children_df.iloc[0]["child_id"]

parent_doc = retrieve_parent_document(
    sample_child_id,
    children_df,
    parents_df,
)

print(parent_doc["content"])

AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrastructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retrieval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encod
